In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a program that performs a 1D convolution operation. Given an input array and a kernel (filter), compute the convolved
  output. The convolution should be performed with a "valid" boundary condition, meaning the kernel is only applied
  where it fully overlaps with the input.
</p>

<svg width="420" height="210" viewBox="0 0 420 210" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;" font-family="monospace" font-size="13">
  <!-- Background -->
  <rect width="420" height="210" rx="8" fill="#222"/>

  <!-- "input" label -->
  <text x="16" y="38" fill="#999" font-size="11">input</text>

  <!-- Input cells -->
  <rect x="65"  y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <rect x="120" y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <rect x="175" y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <rect x="230" y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <rect x="285" y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <!-- Input values -->
  <text x="90"  y="41" text-anchor="middle" fill="#ccc">1</text>
  <text x="145" y="41" text-anchor="middle" fill="#ccc">2</text>
  <text x="200" y="41" text-anchor="middle" fill="#ccc">3</text>
  <text x="255" y="41" text-anchor="middle" fill="#ccc">4</text>
  <text x="310" y="41" text-anchor="middle" fill="#ccc">5</text>

  <!-- Kernel highlight window over first 3 input cells -->
  <rect x="63" y="18" width="164" height="36" rx="4" fill="none" stroke="#4477bb" stroke-width="2" stroke-dasharray="5,3"/>

  <!-- "kernel" label -->
  <text x="16" y="86" fill="#999" font-size="11">kernel</text>

  <!-- Kernel cells (aligned under first 3 input cells) -->
  <rect x="65"  y="68" width="50" height="32" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <rect x="120" y="68" width="50" height="32" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <rect x="175" y="68" width="50" height="32" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <!-- Kernel values -->
  <text x="90"  y="89" text-anchor="middle" fill="#88bbff">1</text>
  <text x="145" y="89" text-anchor="middle" fill="#88bbff">0</text>
  <text x="200" y="89" text-anchor="middle" fill="#88bbff">-1</text>

  <!-- Multiplication signs between pairs -->
  <text x="90"  y="118" text-anchor="middle" fill="#777" font-size="11">1&#xd7;1</text>
  <text x="145" y="118" text-anchor="middle" fill="#777" font-size="11">2&#xd7;0</text>
  <text x="200" y="118" text-anchor="middle" fill="#777" font-size="11">3&#xd7;(-1)</text>

  <!-- Computation line -->
  <text x="145" y="140" text-anchor="middle" fill="#aaa" font-size="12">= 1 + 0 + (-3) = -2</text>

  <!-- Arrow down to output -->
  <line x1="145" y1="148" x2="145" y2="168" stroke="#4477bb" stroke-width="1.5" marker-end="url(#arrowhead)"/>
  <defs>
    <marker id="arrowhead" markerWidth="8" markerHeight="6" refX="8" refY="3" orient="auto">
      <polygon points="0 0, 8 3, 0 6" fill="#4477bb"/>
    </marker>
  </defs>

  <!-- "output" label -->
  <text x="16" y="187" fill="#999" font-size="11">output</text>

  <!-- Output cell -->
  <rect x="120" y="170" width="50" height="30" rx="3" fill="#1a3a1a" stroke="#44aa44" stroke-width="1.5"/>
  <text x="145" y="190" text-anchor="middle" fill="#66dd66" font-weight="bold">-2</text>

  <!-- Ellipsis for remaining output -->
  <text x="195" y="190" fill="#666" font-size="14">&#x2026;</text>
</svg>

<p>
  The input consists of two arrays:
<ul>
  <li><code>input</code>: A 1D array of 32-bit floating-point numbers.</li>
  <li><code>kernel</code>: A 1D array of 32-bit floating-point numbers representing the convolution kernel.</li>
</ul>
The output should be written to the <code>output</code> array, which will have a size of <code>input_size - kernel_size + 1</code>.
</p>

<p>
  The convolution operation is defined mathematically as:
</p>

$$
output[i] = \sum_{j=0}^{kernel\_size-1} input[i + j] \cdot kernel[j]
$$

<p>
  where $i$ ranges from 0 to $input\_size - kernel\_size$.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The
    <code>solve</code> function signature must remain unchanged
  </li>
  <li>The final result must be stored in the array
    <code>output</code>
  </li>
</ul>

<h2>Example 1:</h2>
<pre>
Input: input = [1, 2, 3, 4, 5], kernel = [1, 0, -1]
Output: [-2, -2, -2]
</pre>

<h2>Example 2:</h2>
<pre>
Input: input = [2, 4, 6, 8], kernel = [0.5, 0.2]
Output: [1.8, 3.2, 4.6]
</pre>

<h2>Constraints</h2>

<ul>
  <li>1 &le; <code>input_size</code> &le; 1,500,000</li>
  <li>1 &le; <code>kernel_size</code> &le; 2047</li>
  <li><code>kernel_size</code> &le; <code>input_size</code></li>

  <li>Performance is measured with <code>input_size</code> = 1,500,000, <code>kernel_size</code> = 2,047</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void convolution_1d_kernel(const float* input, const float* kernel, float* output,
                                      int input_size, int kernel_size) {}

// input, kernel, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* input, const float* kernel, float* output, int input_size,
                      int kernel_size) {
    int output_size = input_size - kernel_size + 1;
    int threadsPerBlock = 256;
    int blocksPerGrid = (output_size + threadsPerBlock - 1) / threadsPerBlock;

    convolution_1d_kernel<<<blocksPerGrid, threadsPerBlock>>>(input, kernel, output, input_size,
                                                              kernel_size);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# input, kernel, output are tensors on the GPU
@cute.jit
def solve(
    input: cute.Tensor,
    kernel: cute.Tensor,
    output: cute.Tensor,
    input_size: cute.Int32,
    kernel_size: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# input, kernel are tensors on the GPU
@jax.jit
def solve(input: jax.Array, kernel: jax.Array, input_size: int, kernel_size: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def convolution_1d_kernel(
    input: UnsafePointer[Float32, MutExternalOrigin],
    kernel: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    input_size: Int32,
    kernel_size: Int32,
):
    pass


# input, kernel, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    kernel: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    input_size: Int32,
    kernel_size: Int32,
) raises:
    var output_size = input_size - kernel_size + 1
    var threadsPerBlock: Int32 = 256
    var ctx = DeviceContext()

    var blocksPerGrid = ceildiv(output_size, threadsPerBlock)

    var _kernel = ctx.compile_function[convolution_1d_kernel, convolution_1d_kernel]()
    ctx.enqueue_function(
        _kernel,
        input,
        kernel,
        output,
        input_size,
        kernel_size,
        grid_dim=blocksPerGrid,
        block_dim=threadsPerBlock,
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# input, kernel, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    kernel: torch.Tensor,
    output: torch.Tensor,
    input_size: int,
    kernel_size: int,
):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


@triton.jit
def conv1d_kernel(input, kernel, output, input_size, kernel_size, BLOCK_SIZE: tl.constexpr):
    pass


# input, kernel, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    kernel: torch.Tensor,
    output: torch.Tensor,
    input_size: int,
    kernel_size: int,
):
    BLOCK_SIZE = 1024
    n_blocks = triton.cdiv(input_size - kernel_size + 1, BLOCK_SIZE)
    grid = (n_blocks,)

    conv1d_kernel[grid](input, kernel, output, input_size, kernel_size, BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/easy/9_1d_convolution/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
